In [1]:
import rawpy
import cv2
import numpy as np
import os
import glob
from tqdm import tqdm  # 可选：显示进度条（先 pip install tqdm）

def process_nef(input_path, output_dir=None, half_size=False, use_camera_wb=True):
    """
    处理单个 NEF 文件：YUV 直方图均衡化提升亮度，输出 PNG

    参数:
        input_path: 输入的 .NEF 文件路径
        output_dir: 输出目录（None 表示与输入同目录）
        half_size: 是否缩小一半尺寸
        use_camera_wb: 是否使用相机白平衡
    """
    # 检查输入文件是否存在
    if not os.path.exists(input_path):
        print(f"错误：文件 {input_path} 不存在")
        return False

    try:
        # 1. 用 rawpy 读取 NEF 文件
        with rawpy.imread(input_path) as raw:
            img = raw.postprocess(
                use_camera_wb=use_camera_wb,    # 使用相机白平衡
                half_size=half_size,            # 是否缩小一半
                output_bps=8,                   # 输出 8-bit
                user_flip=0                     # 不翻转
            )
            # 此时 img 是 RGB 顺序 (H, W, 3)
    except Exception as e:
        print(f"  读取失败 {os.path.basename(input_path)}: {e}")
        return False

    # 2. 转换到 YUV 颜色空间（注意是 RGB -> YCrCb）
    imgYUV = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)

    # 3. 分离通道
    channelYUV = cv2.split(imgYUV)

    # 4. 对 Y（亮度）通道做直方图均衡化
    channelYUV[0] = cv2.equalizeHist(channelYUV[0])

    # 5. 合并通道
    channels = cv2.merge(channelYUV)

    # 6. 转换回 RGB 颜色空间
    result = cv2.cvtColor(channels, cv2.COLOR_YCrCb2RGB)

    # 7. 确定输出路径
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    if output_dir is None:
        output_dir = os.path.dirname(input_path)

    os.makedirs(output_dir, exist_ok=True)

    # 输出为 PNG 格式
    output_path = os.path.join(output_dir, f"{base_name}_equalized.png")

    # 8. 保存为 PNG（OpenCV 保存需要 BGR 格式）
    result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, result_bgr, [cv2.IMWRITE_PNG_COMPRESSION, 3])  # 压缩级别 0-9，3 是较好的平衡

    return True


def batch_process_nef(input_dir='img', output_dir='img_processed', half_size=False):
    """
    批量处理目录下所有 NEF 文件，输出 PNG

    参数:
        input_dir: 输入目录（包含 NEF 文件）
        output_dir: 输出目录（默认创建 img_processed）
        half_size: 是否缩小一半尺寸
    """
    # 查找所有 NEF 文件（不区分大小写）
    nef_files = []
    for ext in ['*.NEF', '*.nef']:
        nef_files.extend(glob.glob(os.path.join(input_dir, ext)))

    if not nef_files:
        print(f"在 '{input_dir}' 目录下未找到任何 .NEF 文件")
        return

    print(f"找到 {len(nef_files)} 个 NEF 文件")
    print(f"输出目录: {output_dir}")
    print(f"输出格式: PNG (无损压缩)")
    print("-" * 50)

    # 统计结果
    success_count = 0
    fail_count = 0
    failed_files = []

    # 使用 tqdm 显示进度条（如果已安装）
    try:
        from tqdm import tqdm
        iterator = tqdm(nef_files, desc="处理进度")
    except ImportError:
        iterator = nef_files
        print("提示: 安装 tqdm 可显示进度条 (pip install tqdm)")

    for nef_file in iterator:
        filename = os.path.basename(nef_file)

        # 处理文件
        success = process_nef(
            input_path=nef_file,
            output_dir=output_dir,
            half_size=half_size,
            use_camera_wb=True
        )

        if success:
            success_count += 1
        else:
            fail_count += 1
            failed_files.append(filename)

    # 输出统计结果
    print("-" * 50)
    print(f"✅ 处理完成: {success_count} 个成功, {fail_count} 个失败")

    if failed_files:
        print("\n❌ 失败的文件:")
        for f in failed_files:
            print(f"  - {f}")


# ============ 主程序 ============
if __name__ == "__main__":
    # 批量处理 img 目录下的所有 NEF 文件，输出 PNG
    batch_process_nef(
        input_dir='20240805',            # NEF 文件所在目录
        output_dir='20240805_processed', # 处理后 PNG 输出目录
        half_size=False             # 保持全尺寸（如果内存不足可设为 True）
    )

找到 66 个 NEF 文件
输出目录: 20240805_processed
输出格式: PNG (无损压缩)
--------------------------------------------------


处理进度: 100%|██████████| 66/66 [05:35<00:00,  5.08s/it]

--------------------------------------------------
✅ 处理完成: 66 个成功, 0 个失败


In [2]:
import rawpy
import cv2
import numpy as np
import os
import glob
from tqdm import tqdm  # 可选：显示进度条（先 pip install tqdm）

def process_nef(input_path, output_dir=None, half_size=False, use_camera_wb=True):
    """
    处理单个 NEF 文件：YUV 直方图均衡化提升亮度，输出 PNG

    参数:
        input_path: 输入的 .NEF 文件路径
        output_dir: 输出目录（None 表示与输入同目录）
        half_size: 是否缩小一半尺寸
        use_camera_wb: 是否使用相机白平衡
    """
    # 检查输入文件是否存在
    if not os.path.exists(input_path):
        print(f"错误：文件 {input_path} 不存在")
        return False

    try:
        # 1. 用 rawpy 读取 NEF 文件
        with rawpy.imread(input_path) as raw:
            img = raw.postprocess(
                use_camera_wb=use_camera_wb,    # 使用相机白平衡
                half_size=half_size,            # 是否缩小一半
                output_bps=8,                   # 输出 8-bit
                user_flip=0                     # 不翻转
            )
            # 此时 img 是 RGB 顺序 (H, W, 3)
    except Exception as e:
        print(f"  读取失败 {os.path.basename(input_path)}: {e}")
        return False

    # 2. 转换到 YUV 颜色空间（注意是 RGB -> YCrCb）
    imgYUV = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)

    # 3. 分离通道
    channelYUV = cv2.split(imgYUV)

    # 4. 对 Y（亮度）通道做直方图均衡化
    channelYUV[0] = cv2.equalizeHist(channelYUV[0])

    # 5. 合并通道
    channels = cv2.merge(channelYUV)

    # 6. 转换回 RGB 颜色空间
    result = cv2.cvtColor(channels, cv2.COLOR_YCrCb2RGB)

    # 7. 确定输出路径
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    if output_dir is None:
        output_dir = os.path.dirname(input_path)

    os.makedirs(output_dir, exist_ok=True)

    # 输出为 PNG 格式
    output_path = os.path.join(output_dir, f"{base_name}_equalized.png")

    # 8. 保存为 PNG（OpenCV 保存需要 BGR 格式）
    result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, result_bgr, [cv2.IMWRITE_PNG_COMPRESSION, 3])  # 压缩级别 0-9，3 是较好的平衡

    return True


def batch_process_nef(input_dir='img', output_dir='img_processed', half_size=False):
    """
    批量处理目录下所有 NEF 文件，输出 PNG

    参数:
        input_dir: 输入目录（包含 NEF 文件）
        output_dir: 输出目录（默认创建 img_processed）
        half_size: 是否缩小一半尺寸
    """
    # 查找所有 NEF 文件（不区分大小写）
    nef_files = []
    for ext in ['*.NEF', '*.nef']:
        nef_files.extend(glob.glob(os.path.join(input_dir, ext)))

    if not nef_files:
        print(f"在 '{input_dir}' 目录下未找到任何 .NEF 文件")
        return

    print(f"找到 {len(nef_files)} 个 NEF 文件")
    print(f"输出目录: {output_dir}")
    print(f"输出格式: PNG (无损压缩)")
    print("-" * 50)

    # 统计结果
    success_count = 0
    fail_count = 0
    failed_files = []

    # 使用 tqdm 显示进度条（如果已安装）
    try:
        from tqdm import tqdm
        iterator = tqdm(nef_files, desc="处理进度")
    except ImportError:
        iterator = nef_files
        print("提示: 安装 tqdm 可显示进度条 (pip install tqdm)")

    for nef_file in iterator:
        filename = os.path.basename(nef_file)

        # 处理文件
        success = process_nef(
            input_path=nef_file,
            output_dir=output_dir,
            half_size=half_size,
            use_camera_wb=True
        )

        if success:
            success_count += 1
        else:
            fail_count += 1
            failed_files.append(filename)

    # 输出统计结果
    print("-" * 50)
    print(f"✅ 处理完成: {success_count} 个成功, {fail_count} 个失败")

    if failed_files:
        print("\n❌ 失败的文件:")
        for f in failed_files:
            print(f"  - {f}")


# ============ 主程序 ============
if __name__ == "__main__":
    # 批量处理 img 目录下的所有 NEF 文件，输出 PNG
    batch_process_nef(
        input_dir='20240806',            # NEF 文件所在目录
        output_dir='20240806', # 处理后 PNG 输出目录
        half_size=False             # 保持全尺寸（如果内存不足可设为 True）
    )

找到 60 个 NEF 文件
输出目录: 20240806_processed
输出格式: PNG (无损压缩)
--------------------------------------------------


处理进度: 100%|██████████| 60/60 [05:00<00:00,  5.00s/it]

--------------------------------------------------
✅ 处理完成: 60 个成功, 0 个失败
